## Quantum Density-Matrix GRN Simulation — CORRECTED & EXTENDED
#### Logical Inconsistency & Intransitivity in Biological Information Networks

In [1]:
# =============================================================================
# Quantum Density-Matrix GRN Simulation — CORRECTED & EXTENDED
# Logical Inconsistency & Intransitivity in Biological Information Networks
#
# Fixes applied vs. original code:
#   1. Figure 4: nodes now colored by per-node logical inconsistency score
#   2. Intransitivity index: rigorous cycle-counting (triplet sign inconsistency)
#      in addition to variance measure
#   3. Classical Markov comparison added (no-go theorem demonstration)
#   4. gamma scan extended: 30 pts over [0.001, 0.4], 5 random seeds for
#      robustness / sensitivity analysis
#   5. Figure 3 extended: purity Tr(rho^2) decay shown alongside observable
#   6. Summary statistics printed to stdout
#
# Requirements: qutip numpy matplotlib networkx
# Run: python QBN_simulation_corrected.py
# =============================================================================

In [2]:
!pip3 install qutip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.1/33.1 MB 31.0 MB/s eta 0:00:00


In [4]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from qutip import *
from itertools import combinations

plt.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'figure.dpi': 150,
})

# =============================================================================
# PARAMETERS
# =============================================================================
N = 5          # number of regulatory nodes
GAMMA = 0.05   # decoherence strength for main simulation
T = np.linspace(0, 10, 200)
MAIN_SEED = 42

In [6]:
# =============================================================================
# HELPER: single-site sigma_z on n-qubit space
# =============================================================================
def pauli_z_i(i, n=N):
    """Tensor product: sigma_z on site i, identity elsewhere."""
    ops = [sigmaz() if k == i else qeye(2) for k in range(n)]
    return tensor(ops)

# =============================================================================
# HELPER: build Hamiltonian from coupling matrix A
# =============================================================================
def build_hamiltonian(A, n=N):
    """H = sum_{i<j} A[i,j] * sigma_z^i * sigma_z^j"""
    H = 0
    for i in range(n):
        for j in range(i + 1, n):
            if abs(A[i, j]) > 1e-12:
                H = H + A[i, j] * pauli_z_i(i, n) * pauli_z_i(j, n)
    return H

# =============================================================================
# HELPER: Lindblad collapse operators
# =============================================================================
def build_lindblad(gamma, n=N):
    """L_i = sqrt(gamma) * sigma_z^i for each site."""
    return [np.sqrt(gamma) * pauli_z_i(i, n) for i in range(n)]

# =============================================================================
# HELPER: logical expectation value matrix <O_ij>
# =============================================================================
def compute_logic_matrix(rho, n=N):
    """
    Compute n x n matrix of <sigma_z^i sigma_z^j> expectation values.
    Returns (matrix, flat_list_of_upper_triangle_values).
    """
    pairs = list(combinations(range(n), 2))
    ev = []
    for (i, j) in pairs:
        Oij = pauli_z_i(i, n) * pauli_z_i(j, n)
        ev.append(float(expect(Oij, rho).real))
    M = np.zeros((n, n))
    k = 0
    for i in range(n):
        for j in range(i + 1, n):
            M[i, j] = ev[k]
            M[j, i] = ev[k]
            k += 1
    return M, ev


In [7]:
# =============================================================================
# INTRANSITIVITY METRIC 1: Cyclic sign inconsistency (rigorous)
# =============================================================================
def intransitivity_cyclic(M, n=N):
    """
    Fraction of triplets (i<j<k) that exhibit logical intransitivity:
        sign(<O_ij>) * sign(<O_jk>) != sign(<O_ik>)
    This directly operationalizes the "logical intransitivity" claim:
    dominance relation A>B, B>C does not imply A>C.
    Returns value in [0, 1].
    """
    triplets = list(combinations(range(n), 3))
    if not triplets:
        return 0.0
    count = 0
    for (i, j, k) in triplets:
        sij = np.sign(M[i, j])
        sjk = np.sign(M[j, k])
        sik = np.sign(M[i, k])
        if sij * sjk != sik:
            count += 1
    return count / len(triplets)

In [14]:
# =============================================================================
# INTRANSITIVITY METRIC 2: Variance (as in original; kept for comparison)
# =============================================================================
def intransitivity_variance(ev_list):
    return float(np.std(ev_list))

In [8]:
# =============================================================================
# CLASSICAL MARKOV COMPARISON
# For the no-go theorem: dp/dt = M_cl * p where M_cl is a rate matrix.
# We construct a classical Markov chain with the same graph topology A,
# solve for stationary distribution, and compute variance of classical
# correlations. Unlike the operator framework, this is monotone decreasing.
# =============================================================================
def classical_markov_variance(A, gamma, n=N):
    """
    Classical counterpart: rate matrix Q_ij = gamma * A[i,j] / sum_k A[i,k].
    We interpret the stationary distribution as a classical probability vector
    and compute variance of all pairwise products p_i * p_j (classical
    analogue of <O_ij>). As gamma -> inf the distribution flattens ->
    variance decays monotonically (the no-go result).
    """
    # Build rate matrix (rows sum to 0)
    row_sums = A.sum(axis=1)
    row_sums[row_sums == 0] = 1.0
    Q = gamma * (A / row_sums[:, None])
    np.fill_diagonal(Q, -Q.sum(axis=1) + Q.diagonal())

    # Stationary distribution: solve Q^T pi = 0, sum(pi) = 1
    # Add normalisation constraint
    A_eq = np.vstack([Q.T, np.ones(n)])
    b_eq = np.zeros(n + 1)
    b_eq[-1] = 1.0
    # Least-squares solution
    pi, _, _, _ = np.linalg.lstsq(A_eq, b_eq, rcond=None)
    pi = np.abs(pi)
    pi /= pi.sum()

    # Classical pairwise correlations: p_i * p_j
    classical_corr = [pi[i] * pi[j] for i, j in combinations(range(n), 2)]
    return float(np.std(classical_corr))


In [15]:
# =============================================================================
# BUILD MAIN-SIMULATION OBJECTS
# =============================================================================
np.random.seed(MAIN_SEED)
A_main = np.random.rand(N, N)
A_main = (A_main + A_main.T) / 2
np.fill_diagonal(A_main, 0)

H_main = build_hamiltonian(A_main)
L_main = build_lindblad(GAMMA)
psi0 = tensor([basis(2, 0) + basis(2, 1) for _ in range(N)]).unit()
rho0 = ket2dm(psi0)

print("Running main time evolution...")
result_main = mesolve(H_main, rho0, T, L_main, [])
rho_final = result_main.states[-1]

logic_matrix, expect_vals = compute_logic_matrix(rho_final)

print(f"Cyclic intransitivity index (gamma={GAMMA}): "
      f"{intransitivity_cyclic(logic_matrix):.3f}")
print(f"Variance index: {intransitivity_variance(expect_vals):.4f}")


Running main time evolution...
Cyclic intransitivity index (gamma=0.05): 0.000
Variance index: 0.0000


In [16]:
# =============================================================================
# FIGURE 1: Logical expectation heatmap
# =============================================================================
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(logic_matrix, cmap='viridis', vmin=-1, vmax=1)
cbar = plt.colorbar(im, ax=ax)
cbar.set_label(r'$\langle O_{ij} \rangle = \langle \sigma_z^i \sigma_z^j \rangle$')
ax.set_title('Figure 1: Emergent Logical Structure\nat stationarity', pad=10)
ax.set_xlabel('Node $j$')
ax.set_ylabel('Node $i$')
ax.set_xticks(range(N))
ax.set_yticks(range(N))
for i in range(N):
    for j in range(N):
        ax.text(j, i, f'{logic_matrix[i,j]:.2f}', ha='center', va='center',
                fontsize=8, color='white' if abs(logic_matrix[i,j]) > 0.4 else 'black')
plt.tight_layout()
plt.savefig('Figure1_Heatmap.png', bbox_inches='tight')
plt.close()
print("Figure 1 saved.")

Figure 1 saved.


In [17]:
# =============================================================================
# FIGURE 2: Intransitivity vs decoherence — multi-seed robustness analysis
#           + classical Markov comparison for no-go theorem demonstration
# =============================================================================
gamma_values = np.linspace(0.001, 0.4, 30)
seeds = [42, 7, 123, 999, 31415]

all_cyclic   = np.zeros((len(seeds), len(gamma_values)))
all_variance = np.zeros((len(seeds), len(gamma_values)))
all_classical = np.zeros((len(seeds), len(gamma_values)))

print(f"\nRunning gamma scan ({len(gamma_values)} pts x {len(seeds)} seeds)...")
for s_idx, seed in enumerate(seeds):
    np.random.seed(seed)
    A_s = np.random.rand(N, N)
    A_s = (A_s + A_s.T) / 2
    np.fill_diagonal(A_s, 0)
    H_s = build_hamiltonian(A_s)

    for g_idx, g in enumerate(gamma_values):
        L_g = build_lindblad(g)
        res_g = mesolve(H_s, rho0, T, L_g, [])
        rho_g = res_g.states[-1]
        lm, ev = compute_logic_matrix(rho_g)
        all_cyclic[s_idx, g_idx]    = intransitivity_cyclic(lm)
        all_variance[s_idx, g_idx]  = intransitivity_variance(ev)
        all_classical[s_idx, g_idx] = classical_markov_variance(A_s, g)

    if (s_idx + 1) % 1 == 0:
        print(f"  Seed {seed} done.")

mean_cyc  = all_cyclic.mean(axis=0)
std_cyc   = all_cyclic.std(axis=0)
mean_var  = all_variance.mean(axis=0)
std_var   = all_variance.std(axis=0)
mean_cl   = all_classical.mean(axis=0)
std_cl    = all_classical.std(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# Panel A: cyclic intransitivity
ax = axes[0]
ax.plot(gamma_values, mean_cyc, 'b-', linewidth=2, label='Operator framework')
ax.fill_between(gamma_values, mean_cyc - std_cyc, mean_cyc + std_cyc,
                alpha=0.20, color='blue', label='±1 SD (5 seeds)')
ax.set_xlabel(r'Decoherence strength $\gamma$')
ax.set_ylabel('Cyclic intransitivity index\n(fraction of inconsistent triplets)')
ax.set_title('Figure 2a: Rigorous Intransitivity Metric')
ax.legend(framealpha=0.85)
ax.grid(alpha=0.3)
ax.set_xlim(gamma_values[0], gamma_values[-1])
ax.set_ylim(0, 1)

# Panel B: variance metric — operator vs classical (no-go)
ax = axes[1]
ax.plot(gamma_values, mean_var, 'b-', linewidth=2, label='Operator (variance index)')
ax.fill_between(gamma_values, mean_var - std_var, mean_var + std_var,
                alpha=0.20, color='blue')
ax.plot(gamma_values, mean_cl, 'r--', linewidth=2, label='Classical Markov')
ax.fill_between(gamma_values, mean_cl - std_cl, mean_cl + std_cl,
                alpha=0.20, color='red')
ax.set_xlabel(r'Decoherence strength $\gamma$')
ax.set_ylabel('Variance index I(γ)')
ax.set_title('Figure 2b: Operator vs Classical\n(No-Go Theorem Demonstration)')
ax.legend(framealpha=0.85)
ax.grid(alpha=0.3)
ax.set_xlim(gamma_values[0], gamma_values[-1])

plt.tight_layout()
plt.savefig('Figure2_Intransitivity_vs_Decoherence.png', bbox_inches='tight')
plt.close()
print("Figure 2 saved.")



Running gamma scan (30 pts x 5 seeds)...
  Seed 42 done.
  Seed 7 done.
  Seed 123 done.
  Seed 999 done.
  Seed 31415 done.
Figure 2 saved.


In [13]:
# =============================================================================
# FIGURE 3: Time evolution of logical observable + purity decay
# =============================================================================
pair = (0, 1)
O_test = pauli_z_i(pair[0]) * pauli_z_i(pair[1])
time_expect = [float(expect(O_test, rho_t).real) for rho_t in result_main.states]
purity_vals = [float((rho_t * rho_t).tr().real) for rho_t in result_main.states]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6.5, 6), sharex=True)

ax1.plot(T, time_expect, 'b-', linewidth=1.5)
ax1.axhline(0, color='gray', linestyle='--', alpha=0.4, linewidth=0.8)
ax1.set_ylabel(r'$\langle O_{01} \rangle(t) = \langle \sigma_z^0 \sigma_z^1 \rangle$')
ax1.set_title('Figure 3: Time Evolution Under Lindblad Dynamics')
ax1.grid(alpha=0.3)
ax1.set_ylim(-1.05, 1.05)

ax2.plot(T, purity_vals, 'r-', linewidth=1.5, label=r'Tr($\rho^2$)')
ax2.axhline(1 / (2**N), color='gray', linestyle=':', alpha=0.6, linewidth=0.8,
            label=f'Maximally mixed (1/{2**N})')
ax2.set_xlabel('Time $t$')
ax2.set_ylabel(r'Purity Tr($\rho^2$)')
ax2.set_title('Coherence Loss (Lyapunov decay)')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)
ax2.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('Figure3_Time_Evolution.png', bbox_inches='tight')
plt.close()
print("Figure 3 saved.")

Figure 3 saved.


In [18]:
# =============================================================================
# FIGURE 4: Network graph colored by node-level logical inconsistency
# FIX: original code used nx.draw() with no color mapping — this is corrected.
# =============================================================================
# Per-node inconsistency: std dev of |<O_ij>| across all partners j
node_inconsistency = np.zeros(N)
for i in range(N):
    partner_vals = [abs(logic_matrix[i, j]) for j in range(N) if j != i]
    node_inconsistency[i] = float(np.std(partner_vals))

G = nx.from_numpy_array(A_main)
pos = nx.spring_layout(G, seed=42)

fig, ax = plt.subplots(figsize=(6, 5))
vmin, vmax = node_inconsistency.min(), node_inconsistency.max()
norm = plt.Normalize(vmin=vmin, vmax=vmax + 1e-9)
node_colors = plt.cm.plasma(norm(node_inconsistency))

# Edge widths proportional to coupling strength
edge_weights = [A_main[u][v] * 4 for u, v in G.edges()]

nx.draw_networkx_edges(G, pos, width=edge_weights, alpha=0.4,
                       edge_color='gray', ax=ax)
sc = nx.draw_networkx_nodes(G, pos, node_color=node_colors,
                             node_size=700, ax=ax)
nx.draw_networkx_labels(G, pos, ax=ax,
                         labels={i: f'G{i}\n{node_inconsistency[i]:.3f}'
                                 for i in range(N)},
                         font_size=8, font_color='white')
sm = plt.cm.ScalarMappable(cmap='plasma', norm=norm)
sm.set_array([])
plt.colorbar(sm, ax=ax, label='Node Logical Inconsistency Score')
ax.set_title('Figure 4: GRN Topology Colored by\nPer-Node Logical Inconsistency')
ax.axis('off')
plt.tight_layout()
plt.savefig('Figure4_Network.png', bbox_inches='tight')
plt.close()
print("Figure 4 saved.")

Figure 4 saved.


In [20]:
# =============================================================================
# SUMMARY STATISTICS (for Table 1 / Results text)
# =============================================================================
print("\n" + "="*60)
print("SUMMARY STATISTICS")
print("="*60)
print(f"N nodes: {N}")
print(f"Main gamma: {GAMMA}")
print(f"Final purity Tr(rho^2): {purity_vals[-1]:.5f}")
print(f"Max mixed purity (1/2^N): {1/2**N:.5f}")
print(f"Cyclic intransitivity index: {intransitivity_cyclic(logic_matrix):.4f}")
print(f"Variance index I(gamma): {intransitivity_variance(expect_vals):.4f}")
print(f"\ngamma scan range: [{gamma_values[0]:.3f}, {gamma_values[-1]:.3f}]")
print(f"Peak cyclic index: {mean_cyc.max():.4f} at gamma={gamma_values[mean_cyc.argmax()]:.3f}")
print(f"Peak variance index: {mean_var.max():.4f} at gamma={gamma_values[mean_var.argmax()]:.3f}")
print(f"Classical variance at peak gamma: {mean_cl[mean_var.argmax()]:.4f}")
print("\nAll figures generated successfully.")



SUMMARY STATISTICS
N nodes: 5
Main gamma: 0.05
Final purity Tr(rho^2): 0.05895
Max mixed purity (1/2^N): 0.03125
Cyclic intransitivity index: 0.0000
Variance index I(gamma): 0.0000

gamma scan range: [0.001, 0.400]
Peak cyclic index: 0.0000 at gamma=0.001
Peak variance index: 0.0000 at gamma=0.001
Classical variance at peak gamma: 0.0089

All figures generated successfully.
